# Pipeline

## Setup

In [ ]:
import random
import os
from typing import Callable
import hashlib
import inspect
import traceback
import itertools
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from tqdm import tqdm
import IPython.display as ipd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
import tensorflow as tf
import librosa
import params
from IPython.display import clear_output

In [ ]:
random.seed(42)
np.random.seed(42)

In [ ]:
try:
    os.makedirs("cache")
except FileExistsError:
    pass

In [ ]:
assert len(tf.config.list_physical_devices('GPU'))

## Data

Load data from files

In [ ]:
def load_file(file_path: str) -> np.ndarray:
    # Load file
    audio, _ = librosa.load(file_path, sr=params.sample_rate)

    target_length = params.sample_rate * 30

    # zero padding
    if len(audio) < target_length:
        padded_audio = np.pad(audio, (0, target_length - len(audio)), "constant")
    else:
        padded_audio = audio[:target_length]

    assert len(padded_audio) == 661500
    return padded_audio

In [ ]:
# load metadata
tracks = pd.read_csv("data/fma_metadata/tracks.csv", index_col=0, header=[0, 1])

# create a dataframe with just those files that are part of the small subset
builder = {"path": [], "genre": [], "track_id": []}

for folder in os.listdir("data/fma_small"):
    if not os.path.isdir(os.path.join("data/fma_small", folder)):
        continue
    for file in os.listdir(os.path.join("data/fma_small", folder)):
        if not file.endswith(".mp3"):
            continue
        # iterate over all existing files (we only use a subset of the full dataset)
        # and add them to our dataframe along with their genre
        track_id = int(file.removesuffix(".mp3"))
        genre = tracks.loc[track_id, ("track", "genre_top")]
        builder["track_id"].append(track_id)
        builder["path"].append(os.path.join("data/fma_small", folder, file))
        builder["genre"].append(genre)

# build the dataframe
df = pd.DataFrame(builder)
del builder

# create a consistent random order
permutation = np.load("permutation.npy")
valid_permutation = permutation[permutation < len(df)]

df = df.iloc[valid_permutation].reset_index(drop=True)

# sanity checks
assert all([l > 950 for l in df["genre"].value_counts().values])
assert len(df) > 950 * 8
assert len(df["genre"].unique()) == 8

# manipulate
unique_genres = sorted(list(set(df["genre"])))
n_genres = len(unique_genres)
assert n_genres == 8
genre_to_id = {unique_genres[_]: _ for _ in range(n_genres)}
id_to_genre = {_: unique_genres[_] for _ in range(n_genres)}
df["genre_id"] = df["genre"].map(genre_to_id)

Cache Management

In [ ]:
def compute_hash(data) -> str:
    sha256_hash = hashlib.sha256()
    sha256_hash.update(str(data).encode("utf-8"))
    return sha256_hash.hexdigest()

In [ ]:
def split_dataset(
    df: pd.DataFrame, train_size: float, test_size: float
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # split
    train_size = int(len(df) * train_size)
    test_size = int(len(df) * test_size)
    val_size = len(df) - train_size - test_size
    assert train_size > 10
    assert test_size > 10
    assert val_size > 10
    assert len(df) == train_size + test_size + val_size

    df_train = df.iloc[:train_size]
    df_test = df.iloc[train_size : train_size + test_size]
    df_val = df.iloc[train_size + test_size :]

    assert len(df) == len(df_train) + len(df_test) + len(df_val)
    print(
        f"Train size: {len(df_train)} Test size: {len(df_test)} Val size: {len(df_val)}"
    )
    return df_train, df_test, df_val

In [ ]:
def build_targets_features(
    data: pd.DataFrame, splits: int, feature_extractor: Callable, desc=str
) -> tuple[np.ndarray, np.ndarray]:

    features = []
    targets = []
    for _, row in tqdm(
        data.iterrows(), total=len(data), desc=f"Building {desc} dataset"
    ):
        raw = load_file(row["path"])
        assert not len(raw) % splits
        step = int(len(raw) / splits)
        for i in range(splits):
            raw_chunk = raw[i * step : (i + 1) * step]
            features.append(feature_extractor(raw_chunk))
            targets.append(row["genre_id"])

    X = np.vstack(features)
    y = np.array(targets).flatten()

    assert len(X) == len(data) * splits
    assert len(y) == len(data) * splits
    idx = np.random.permutation(len(X))
    return X[idx], y[idx]

In [ ]:
def get_train_test_val_features_and_targets(
    df: pd.DataFrame,
    feature_extractor: Callable,
    splits: int,
    train_size: float,
    test_size: float,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    data = load_file(df.loc[0, "path"])
    source_code = inspect.getsource(feature_extractor)
    identifier = compute_hash(
        (
            feature_extractor(data),
            feature_extractor.__name__,
            source_code,
            splits,
            train_size,
            test_size,
            len(df),
        )
    )

    if os.path.isdir(os.path.join("cache", identifier)):
        return (
            (
                np.load(os.path.join("cache", identifier, "X_train.npy")),
                np.load(os.path.join("cache", identifier, "y_train.npy")),
            ),
            (
                np.load(os.path.join("cache", identifier, "X_test.npy")),
                np.load(os.path.join("cache", identifier, "y_test.npy")),
            ),
            (
                np.load(os.path.join("cache", identifier, "X_val.npy")),
                np.load(os.path.join("cache", identifier, "y_val.npy")),
            ),
        )

    else:
        df_train, df_test, df_val = split_dataset(
            df=df, train_size=train_size, test_size=test_size
        )

        # create data for model
        X_train, y_train = build_targets_features(
            data=df_train,
            splits=splits,
            feature_extractor=feature_extractor,
            desc="train",
        )
        X_test, y_test = build_targets_features(
            data=df_test,
            splits=splits,
            feature_extractor=feature_extractor,
            desc="test",
        )
        X_val, y_val = build_targets_features(
            data=df_val, splits=splits, feature_extractor=feature_extractor, desc="val"
        )
        assert len(X_train) + len(X_test) + len(X_val) == len(df) * splits
        assert len(y_train) + len(y_test) + len(y_val) == len(df) * splits

        # cache for future use
        os.makedirs(os.path.join("cache", identifier))
        np.save(os.path.join("cache", identifier, "X_train.npy"), X_train)
        np.save(os.path.join("cache", identifier, "y_train.npy"), y_train)
        np.save(os.path.join("cache", identifier, "X_test.npy"), X_test)
        np.save(os.path.join("cache", identifier, "y_test.npy"), y_test)
        np.save(os.path.join("cache", identifier, "X_val.npy"), X_val)
        np.save(os.path.join("cache", identifier, "y_val.npy"), y_val)

        return (X_train, y_train), (X_test, y_test), (X_val, y_val)

Utilization Functions

In [ ]:
class LivePlot(tf.keras.callbacks.Callback):
    def __init__(self, logy=False):
        super().__init__()
        self.logy = logy
        self.train_loss = []
        self.val_loss = []
        self.train_acc = []
        self.val_acc = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        # Collect metrics
        self.train_loss.append(logs.get("loss"))
        self.val_loss.append(logs.get("val_loss"))
        self.train_acc.append(logs.get("accuracy"))
        self.val_acc.append(logs.get("val_accuracy"))

        # Update plot
        clear_output(wait=True)
        plt.figure(figsize=(12, 5))

        # loss
        plt.subplot(1, 2, 1)
        plt.plot(self.train_loss, label="Train Loss")
        plt.plot(self.val_loss, label="Validation Loss")
        plt.title("Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss" + " (Log Scale)" if self.logy else "Loss")
        if self.logy:
            plt.yscale("log")
        plt.legend()
        plt.grid(True, which="both")

        # accuracy
        plt.subplot(1, 2, 2)
        plt.plot(self.train_acc, label="Train Accuracy")
        plt.plot(self.val_acc, label="Validation Accuracy")
        plt.axhline(
            y=1 / n_genres,
            color="red",
            linestyle="--",
            linewidth=2,
            label="Random Threshold",
        )
        plt.title("Accuracy")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.legend()
        plt.grid(True)

        plt.tight_layout()
        plt.show()

In [ ]:
def show_mel_spectrogram(index: int):
    mel_spectrogram = librosa.feature.mfcc(
        y=load_file(df.loc[index, "path"]), sr=params.sample_rate, n_mels=40
    )
    mel_spectrogram_db = librosa.power_to_db(mel_spectrogram, ref=np.max)

    plt.figure(figsize=(10, 4))
    librosa.display.specshow(
        mel_spectrogram_db,
        sr=params.sample_rate,
        x_axis="time",
        y_axis="mel",
        cmap="coolwarm",
    )
    plt.colorbar(format="%+2.0f dB")
    plt.tight_layout()
    plt.title(
        f'Mel Spectrogram for audio at index {index}, genre: {df.iloc[index]["genre"]}'
    )
    plt.show()


def play_audio(index: int):
    ipd.display(
        ipd.Audio(data=load_file(df.loc[index, "path"]), rate=params.sample_rate)
    )


def investigate(index: int):
    show_mel_spectrogram(index)
    play_audio(index)

In [ ]:
investigate(np.random.randint(0, len(df) - 1))

## Pipeline

In [ ]:
def pipeline(
    df: pd.DataFrame,
    train_size: float,
    test_size: float,
    splits: int,
    feature_extractor: Callable,
    model_creator: Callable,
    epochs: int,
    batch_size: int,
    earlystop_patience: int,
    learning_rate: int,
):
    (X_train, y_train), (X_test, y_test), (X_val, y_val) = (
        get_train_test_val_features_and_targets(
            df=df,
            feature_extractor=feature_extractor,
            splits=splits,
            train_size=train_size,
            test_size=test_size,
        )
    )
    assert len(X_train) == len(y_train)
    assert len(X_test) == len(y_test)
    assert len(X_val) == len(y_val)
    assert len(X_train) + len(X_test) + len(X_val) == len(df) * splits
    print(f"Train size: {len(X_train)}")
    print(f"Test size: {len(X_test)}")
    print(f"Validation size: {len(X_val)}")

    scaler = StandardScaler().fit(X_train)

    x_train_transformed = scaler.transform(X_train)
    x_val_transformed = scaler.transform(X_val)
    x_test_transformed = scaler.transform(X_test)

    x_train_transformed = tf.convert_to_tensor(x_train_transformed, dtype=tf.float32)
    y_train = tf.convert_to_tensor(y_train, dtype=tf.int32)

    # create model
    model = model_creator(n_features=x_train_transformed.shape[1])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],  # loss is implicitly included
    )
    model.summary()

    with tf.device("/GPU:0"):
        # fit model
        history = model.fit(
            x_train_transformed,
            y_train,
            epochs=epochs,
            batch_size=batch_size,
            validation_data=(x_val_transformed, y_val),
            callbacks=[
                tf.keras.callbacks.EarlyStopping(
                    monitor="val_accuracy",
                    patience=earlystop_patience,
                    restore_best_weights=True,
                ),
                LivePlot(logy=True),
            ],
            verbose=0,
        )

    # calculate loss & accuracy
    logits = model(x_test_transformed)
    y_pred = np.argmax(logits, axis=1)

    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    loss = loss_fn(y_test, logits).numpy()

    accuracy = tf.reduce_mean(tf.cast(y_pred == y_test, tf.float32)).numpy()

    # confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(10, 7))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=unique_genres,
        yticklabels=unique_genres,
    )

    plt.title(f"Confusion Matrix. Accuracy={accuracy*100:.2f}%")
    plt.xlabel("Predicted Genre")
    plt.ylabel("True Genre")
    plt.show()
    return model, history, loss, accuracy

# Configuration

## Features

In [ ]:
def compute_mel_spectrogram(y: np.ndarray):
    mel_spectrogram = librosa.feature.mfcc(y=y, sr=params.sample_rate, n_mels=40)
    mel_spectrogram_db = librosa.power_to_db(mel_spectrogram, ref=np.max)
    return mel_spectrogram_db


def compute_spectral_centroid(y: np.ndarray):
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=params.sample_rate)
    return spectral_centroid


def compute_spectral_bandwidth(y: np.ndarray):
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=params.sample_rate)
    return spectral_bandwidth


def compute_spectral_rolloff(y: np.ndarray):
    spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=params.sample_rate)
    return spectral_rolloff


def compute_spectral_contrast(y: np.ndarray):
    spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=params.sample_rate)
    return spectral_contrast


def compute_spectral_flatness(y: np.ndarray):
    spectral_flatness = librosa.feature.spectral_flatness(y=y)
    return spectral_flatness


def compute_spectral_features(y: np.ndarray):
    features = [
        compute_mel_spectrogram(y),
        compute_spectral_centroid(y),
        compute_spectral_bandwidth(y),
        compute_spectral_rolloff(y),
        compute_spectral_contrast(y),
        compute_spectral_flatness(y),
    ]
    return np.vstack(features).flatten()

## Model

In [ ]:
def create_simple_model(n_features: int) -> tf.keras.Model:
    """Simple MLP model"""
    inp = tf.keras.layers.Input(shape=(n_features,))
    x = tf.keras.layers.Dense(256, activation="relu")(inp)
    x = tf.keras.layers.Dropout(0.5)(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    out = tf.keras.layers.Dense(n_genres, activation="softmax")(x)
    model = tf.keras.Model(inputs=inp, outputs=out)
    return model

## Run

In [ ]:
splits_list = [10, 6, 3, 1]
batch_sizes = [16, 32, 64, 128, 256]
learning_rates = [0.01, 0.001, 0.0001, 1e-05, 1e-06, 1e-07]
feature_extractors = [compute_spectral_features]
model_creators = [create_simple_model]

combinations = list(
    itertools.product(
        splits_list, batch_sizes, learning_rates, feature_extractors, model_creators
    )
)

output_file = "grid_search_results.csv"

if os.path.exists(output_file):
    results_df = pd.read_csv(output_file)
else:
    results_df = pd.DataFrame(
        columns=[
            "splits",
            "batch_size",
            "learning_rate",
            "feature_extractor",
            "model_creator",
            "accuracy",
            "loss",
        ]
    )

completed = set(
    tuple(x)
    for x in results_df[
        ["splits", "batch_size", "learning_rate", "feature_extractor", "model_creator"]
    ].values
)

for splits, batch_size, lr, feature_extractor, model_creator in combinations:
    if (
        splits,
        batch_size,
        lr,
        feature_extractor.__name__,
        model_creator.__name__,
    ) in completed:
        continue

    print(f"Running: splits={splits}, batch_size={batch_size}, lr={lr}")
    model, history, loss, accuracy = pipeline(
        df=df,
        train_size=0.6,
        test_size=0.2,
        splits=splits,
        feature_extractor=feature_extractor,
        model_creator=model_creator,
        epochs=100,
        batch_size=batch_size,
        earlystop_patience=10,
        learning_rate=lr,
    )

    results_df = pd.concat(
        [
            results_df,
            pd.DataFrame(
                [
                    {
                        "splits": splits,
                        "batch_size": batch_size,
                        "learning_rate": lr,
                        "feature_extractor": feature_extractor.__name__,
                        "model_creator": model_creator.__name__,
                        "loss": loss,
                        "accuracy": accuracy,
                    }
                ]
            ),
        ],
        ignore_index=True,
    )

    results_df.to_csv(output_file, index=False)